# Notebook 01: Data Ingestion & Preprocessing

This notebook handles:
1. Downloading the DBLP XML dataset from the official server
2. Parsing the XML using a streaming approach
3. Saving processed data to Parquet for fast reuse

---

## Subset Strategy

The full DBLP dataset contains **~8.4M publications**. We will use the full data for the EDA part (with some small modifications) and sample smaller subsets when needed (time or memory issues).

In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: d:\Rekrutacja_nokia


## 1. Download DBLP Dataset

We download `dblp.xml.gz` (~1 GB) and `dblp.dtd` from the official DBLP server.
The DTD file is required for the XML parser to resolve entity references.

In [2]:
import urllib.request
import time

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

DBLP_XML_URL = "https://dblp.org/xml/dblp.xml.gz"
DBLP_DTD_URL = "https://dblp.org/xml/dblp.dtd"

xml_path = DATA_DIR / "dblp.xml.gz"
dtd_path = DATA_DIR / "dblp.dtd"

def download_file(url: str, dest: Path, description: str = ""):
    """Download a file with progress reporting."""
    if dest.exists():
        size_mb = dest.stat().st_size / (1024 * 1024)
        print(f"{description} already exists ({size_mb:.1f} MB): {dest}")
        return
    
    print(f"Downloading {description} from {url}...")
    start = time.time()
    
    def progress_hook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            pct = downloaded / total_size * 100
            mb_down = downloaded / (1024*1024)
            mb_total = total_size / (1024*1024)
            print(f"\r   {mb_down:.1f}/{mb_total:.1f} MB ({pct:.1f}%)", end="", flush=True)
    
    urllib.request.urlretrieve(url, dest, reporthook=progress_hook)
    elapsed = time.time() - start
    size_mb = dest.stat().st_size / (1024 * 1024)
    print(f"\nDownloaded {size_mb:.1f} MB in {elapsed:.0f}s")

# Download DTD first, small but has metadata required for parsing https://dblp.uni-trier.de/xml/docu/dblpxml.pdf
download_file(DBLP_DTD_URL, dtd_path, "dblp.dtd")

# Download the main dataset
download_file(DBLP_XML_URL, xml_path, "dblp.xml.gz")

dblp.dtd already exists (0.0 MB): d:\Rekrutacja_nokia\data\dblp.dtd
dblp.xml.gz already exists (1011.3 MB): d:\Rekrutacja_nokia\data\dblp.xml.gz


## 2. Parse DBLP XML

We use `lxml.iterparse` to stream through the compressed XML file.

This approach allows us to filter and handle edgecases while loading the data. The whole ~8,4 M database takes up about 2GB of RAM when saved to pandas. DataFrame, so we can load everything at the start and then sample accordingly.

In [3]:
from src.parser import parse_dblp_xml, save_to_parquet

MIN_YEAR = 1935

print(f"Parsing DBLP XML with year >= {MIN_YEAR}...")
print(f"Source: {xml_path}")
print(f"DTD: {dtd_path}")
print()

df = parse_dblp_xml(
    xml_path,
    min_year=MIN_YEAR,
    max_records=None,  # Set to 10000 for testing
    show_progress=True,
)

Parsing DBLP XML with year >= 1935...
Source: d:\Rekrutacja_nokia\data\dblp.xml.gz
DTD: d:\Rekrutacja_nokia\data\dblp.dtd


Parsed 8,462,457 records (skipped 0 records before 1935)


## 3. Initial Data Exploration

Let's look at what we parsed before saving.

In [4]:
print(f"=== Dataset Overview ===")
print(f"Total records: {len(df):,}")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

=== Dataset Overview ===
Total records: 8,462,457
Year range: 1936 - 2026

Columns: ['title', 'authors', 'year', 'venue', 'pub_type', 'venue_category', 'key']

Data types:
title                  str
authors             object
year                 Int64
venue                  str
pub_type          category
venue_category    category
key                    str
dtype: object

Memory usage: 2061.5 MB


In [5]:
print("=== Sample Records ===")
df.head(10)

=== Sample Records ===


,title,authors,year,venue,pub_type,venue_category,key
0,Curvedness.,[],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/X14bd
1,Transparency and Translucency.,[Manish Singh 0001],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/Singh14
2,Surface Orientation Histogram (Discrete Versio...,[],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/X14ii
3,Appearance Scanning.,[],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/X14m
4,Image-Based Lighting.,[Tien-Tsin Wong],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/Wong14
5,Radiometric Camera Calibration.,[],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/X14gt
6,Surface Roughness.,[Sylvia C. Pont],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/Pont14b
7,Image Registration.,[Daniel C. Alexander],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/Alexander14
8,Subspace Methods.,[Kazuhiro Fukui],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/Fukui14
9,Structure-from-Motion (SfM).,[],2014,"Computer Vision, A Reference Guide",incollection,other,reference/vision/X14if


In [6]:
import pandas as pd

print("=== Missing Values ===")
null_counts = df.isnull().sum()
null_pcts = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'null_count': null_counts,
    'null_percent': null_pcts
})
print(missing_df)

# Check authors column specifically
empty_authors = df['authors'].apply(lambda x: len(x) == 0 if isinstance(x, list) else True).sum()
print(f"\nRecords with no authors: {empty_authors:,} ({empty_authors/len(df)*100:.1f}%)")

=== Missing Values ===
                null_count  null_percent
title                    0          0.00
authors                  0          0.00
year                     3          0.00
venue               172846          2.04
pub_type                 0          0.00
venue_category           0          0.00
key                      0          0.00

Records with no authors: 112,542 (1.3%)


In [7]:
# Publication type distribution
print("=== Publication Types ===")
type_counts = df['pub_type'].value_counts()
for ptype, count in type_counts.items():
    print(f"  {ptype:20s}: {count:>10,} ({count/len(df)*100:.1f}%)")

print(f"\n=== Venue Category ===")
cat_counts = df['venue_category'].value_counts()
for cat, count in cat_counts.items():
    print(f"  {cat:20s}: {count:>10,} ({count/len(df)*100:.1f}%)")

=== Publication Types ===
  article             :  4,280,730 (50.6%)
  inproceedings       :  3,872,430 (45.8%)
  phdthesis           :    152,755 (1.8%)
  incollection        :     71,095 (0.8%)
  proceedings         :     64,044 (0.8%)
  book                :     21,376 (0.3%)
  mastersthesis       :         27 (0.0%)

=== Venue Category ===
  journal             :  4,280,730 (50.6%)
  conference          :  3,936,474 (46.5%)
  other               :    245,253 (2.9%)


## 4. Save Processed Data

We save the processed DataFrame to Parquet format for fast loading in subsequent notebooks. ~20s loading for 8,4M lines of data.


In [14]:
from src.parser import save_to_parquet

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = PROCESSED_DIR / "dblp_subset.parquet"
save_to_parquet(df, parquet_path)

print(f"\nData ingestion complete!")
print(f"Records: {len(df):,}")
print(f"Saved to: {parquet_path}")

Saved 8,462,457 records to d:\Rekrutacja_nokia\data\processed\dblp_subset.parquet (794.2 MB)

Data ingestion complete!
Records: 8,462,457
Saved to: d:\Rekrutacja_nokia\data\processed\dblp_subset.parquet


In [10]:
from src.parser import load_from_parquet

df_verify = load_from_parquet(parquet_path)
assert len(df_verify) == len(df), "Parquet load size mismatch!"
assert isinstance(df_verify['authors'].iloc[0], list), "Authors should be a list!"
print(f"Verification passed: {len(df_verify):,} records loaded from parquet")
print(f"Sample authors: {df_verify['authors'].iloc[1000]}")

Verification passed: 8,462,457 records loaded from parquet
Sample authors: ['Arthur Prochazka']
